# 01 - Preprocessing: clean raw CSVs and define the target

Loads `loan`, `trans`, `client`, `account`, `disp` from `../data`, checks quality
(duplicates, nulls, date parsing), derives age/gender from birth numbers,
and maps loan `status` to the binary target: A/C = good (0), B/D = default (1).

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA_DIR = ROOT / "data"
assert DATA_DIR.exists(), f"data dir not found from {Path.cwd()}"

dfs = {n: pd.read_csv(DATA_DIR / f"{n}.csv", sep=";", low_memory=False) for n in ["loan", "trans", "client", "account", "disp"]}
loan, trans, client, account, disp = (dfs[n] for n in ["loan", "trans", "client", "account", "disp"])

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

loan: (682, 7)
trans: (1056320, 10)
client: (5369, 3)
account: (4500, 4)
disp: (5369, 4)


In [2]:
for name, df in dfs.items():
    print(f"--- {name} ---")
    print("duplicates:", df.duplicated().sum())
    nulls = df.isnull().sum()
    print(nulls[nulls > 0].to_string() if (nulls > 0).any() else "no nulls")

--- loan ---
duplicates: 0
no nulls
--- trans ---


duplicates: 0
operation    183114
k_symbol     481881
bank         782812
account      760931
--- client ---
duplicates: 0
no nulls
--- account ---
duplicates: 0
no nulls
--- disp ---
duplicates: 0
no nulls


In [3]:
for df, col in [(trans, "date"), (loan, "date")]:
    parsed = pd.to_datetime(df[col].astype(str), format="%y%m%d", errors="coerce")
    print(col, "unparseable:", parsed.isna().sum(), "| range:", parsed.min().date(), "->", parsed.max().date())

date unparseable: 0 | range: 1993-01-01 -> 1998-12-31
date unparseable: 0 | range: 1993-07-05 -> 1998-12-08


In [4]:
bn = client["birth_number"].astype(str).str.zfill(6)
month = bn.str[2:4].astype(int)
client["gender"] = (month > 50).map({True: "Female", False: "Male"})
client["age"] = 1999 - (bn.str[:2].astype(int) + 1900)
print(client.groupby("gender")["age"].describe().round(1).to_string())

         count  mean   std   min   25%   50%   75%   max
gender                                                  
Female  2645.0  45.4  17.1  12.0  31.0  45.0  58.0  85.0
Male    2724.0  46.2  17.4  13.0  32.0  46.0  59.2  88.0


In [5]:
loan["target"] = loan["status"].map({"A": 0, "C": 0, "B": 1, "D": 1})
print(loan["target"].value_counts().rename({0: "good (0)", 1: "default (1)"}).to_string())
print(f"default rate: {loan['target'].mean():.2%}")

target
good (0)       606
default (1)     76
default rate: 11.14%


In [6]:
assert loan.shape[0] == 682, loan.shape
assert loan["status"].isin(["A", "B", "C", "D"]).all(), loan["status"].unique()
assert loan["target"].isna().sum() == 0, "unknown status mapped to NaN"
assert set(loan["target"].unique()) <= {0, 1}
assert loan["target"].sum() == ((loan["status"] == "B") | (loan["status"] == "D")).sum()
assert client["age"].between(0, 100).all()
print("preprocessing checks passed")

preprocessing checks passed
